# Reviewer Robustness Checks — A100

Implements the Major-Revision robustness checks without retraining any Transformer.

- **A1:** source-supported counterfactual intervention on existing scratch checkpoints.
- **A2:** 10 independent token maps on the same frozen Qwen checkpoint/source.
- **A3:** probe capacity × training-set-size sweep with a matched true-state control.
- **A4:** stationary-weighted \(H_{k-1}-H^*\) audit and seed-level stability.

All outputs are written to a new `Markov_Context_Paper/Reviewer_Robustness_Checks/` directory. Existing runs/checkpoints are read-only and every heavy section is resumable.

In [ ]:
import os, sys, json, math, time, random, hashlib, gc, zipfile
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoModelForCausalLM

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16=bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
MODEL_DTYPE=torch.bfloat16 if USE_BF16 else torch.float16
BASE=Path.cwd()/'Markov_Context_Paper'
PART1=BASE/'Part1'; PART2=BASE/'Part2'
OUT=BASE/'Reviewer_Robustness_Checks'; CSV_DIR=OUT/'csv'; FIG_DIR=OUT/'figures'; CACHE_DIR=PART2/'hf_cache'
for p in [OUT,CSV_DIR,FIG_DIR,CACHE_DIR]: p.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME']=str(CACHE_DIR)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('dtype:',MODEL_DTYPE)
print('OUT:',OUT)


In [ ]:
V=8; KS=[2,3,4]; SCRATCH_SEEDS=[0,1,2,3,4]
A1_N_CONTEXTS=20_000; A1_BATCH=2_000; CHAIN_WARMUP=256
MODELS={'qwen05':'Qwen/Qwen2.5-0.5B','qwen15':'Qwen/Qwen2.5-1.5B'}
A2_SOURCE_SEED=0; A2_MAP_SEEDS=list(range(10))
A2_N_TRAIN=12_000; A2_N_VAL=3_000; A2_N_TEST=6_000
EXTRACT_BATCH={'qwen05':256,'qwen15':128}
A3_KS=[3,4]; A3_SOURCE_SEED=0; A3_MAP_SEED=0; A3_FRACTIONS=[0.25,0.50,1.00]
PROBE_BATCH=512; PATIENCE=4; LINEAR_EPOCHS=25; MLP_EPOCHS=30
LINEAR_LR=1e-3; MLP_LR=5e-4; PROBE_WD=1e-4
QUICK_TEST=False
if QUICK_TEST:
    A1_N_CONTEXTS=2_000; A1_BATCH=500; A2_MAP_SEEDS=[0,1]
    A2_N_TRAIN,A2_N_VAL,A2_N_TEST=1000,500,500
    A3_FRACTIONS=[0.5,1.0]; LINEAR_EPOCHS=3; MLP_EPOCHS=3


In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def clean_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def save_json(p,o):
    p.parent.mkdir(parents=True,exist_ok=True)
    with open(p,'w',encoding='utf-8') as f:
        json.dump(o,f,indent=2,default=lambda x: float(x) if isinstance(x,np.floating) else int(x) if isinstance(x,np.integer) else str(x))

def trusted_torch_load(path,map_location=DEVICE):
    try: return torch.load(path,map_location=map_location,weights_only=False)
    except TypeError: return torch.load(path,map_location=map_location)

save_json(OUT/'environment.json',{'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'dtype':str(MODEL_DTYPE),'models':MODELS})


## Shared Markov utilities
Stationary probabilities are computed directly on the \(V^k\) shift-register chain by power iteration.

In [ ]:
def context_to_index(c,V):
    idx=torch.zeros(c.shape[:-1],dtype=torch.long,device=c.device)
    for j in range(c.shape[-1]): idx=idx*V+c[...,j]
    return idx

def generate_kernel(V,k,seed,device=DEVICE):
    g=torch.Generator(device=device); g.manual_seed(seed+10000019)
    return torch.softmax(torch.randn((V**k,V),generator=g,device=device),dim=-1)

class PersistentMarkovSampler:
    def __init__(self,P,V,k,batch_size,max_history,seed,warmup_steps=256):
        self.P=P; self.V=V; self.k=k; self.batch_size=batch_size; self.max_history=max(max_history,k)
        self.gen=torch.Generator(device=DEVICE); self.gen.manual_seed(seed+4000003)
        self.history=torch.randint(0,V,(batch_size,self.max_history),generator=self.gen,device=DEVICE)
        for _ in range(warmup_steps): self.advance()
    @torch.no_grad()
    def draw_next(self):
        probs=self.P[context_to_index(self.history[:,-self.k:],self.V)]
        return torch.multinomial(probs,1,generator=self.gen).squeeze(1)
    @torch.no_grad()
    def advance(self):
        y=self.draw_next(); self.history=torch.cat([self.history[:,1:],y[:,None]],1); return y
    @torch.no_grad()
    def sample(self,L):
        x=self.history[:,-L:].clone(); y=self.draw_next(); self.history=torch.cat([self.history[:,1:],y[:,None]],1); return x,y

@torch.no_grad()
def stationary_distribution(P,V,k,tol=1e-13,max_iter=200000):
    P64=P.detach().double().to(DEVICE); n=V**k
    pi=torch.full((n,),1.0/n,dtype=torch.float64,device=DEVICE)
    states=torch.arange(n,device=DEVICE,dtype=torch.long)
    suffix=states%(V**(k-1)) if k>1 else torch.zeros_like(states)
    next_idx=suffix[:,None]*V+torch.arange(V,device=DEVICE)[None,:]
    for it in range(max_iter):
        new=torch.zeros_like(pi); new.scatter_add_(0,next_idx.reshape(-1),(pi[:,None]*P64).reshape(-1))
        new=new/new.sum().clamp_min(1e-300)
        err=torch.max(torch.abs(new-pi)).item(); pi=new
        if err<tol: break
    else: raise RuntimeError(f'stationary iteration failed k={k}')
    return pi/pi.sum(),it+1,err

@torch.no_grad()
def exact_stationary_information(P,V,k):
    P64=P.detach().double().to(DEVICE); pi,n_iter,res=stationary_distribution(P64,V,k)
    row_H=-(P64*torch.log(P64.clamp_min(1e-300))).sum(-1)
    Hs=float((pi*row_H).sum())
    n_suffix=V**(k-1); states=torch.arange(V**k,device=DEVICE); suffix=states%n_suffix
    mass=torch.zeros(n_suffix,dtype=torch.float64,device=DEVICE)
    joint=torch.zeros((n_suffix,V),dtype=torch.float64,device=DEVICE)
    mass.scatter_add_(0,suffix,pi); joint.index_add_(0,suffix,pi[:,None]*P64)
    q=joint/mass[:,None].clamp_min(1e-300); qstate=q[suffix].clamp_min(1e-300)
    Hkm1=float((pi*(-(P64*torch.log(qstate)).sum(-1))).sum())
    return {'H_star':Hs,'H_km1':Hkm1,'gap':Hkm1-Hs,'stationary_iters':n_iter,'stationary_residual':res}

@torch.no_grad()
def oldest_given_suffix(pi,V,k):
    tab=pi.reshape(V,V**(k-1)).T.contiguous()
    return tab/tab.sum(-1,keepdim=True).clamp_min(1e-300)

@torch.no_grad()
def generate_dataset(P,V,k,L,n,seed,batch_size=1024):
    g=torch.Generator(device=DEVICE); g.manual_seed(seed+7000009)
    B=min(batch_size,n); hlen=max(k,L); hist=torch.randint(0,V,(B,hlen),generator=g,device=DEVICE)
    for _ in range(256):
        probs=P[context_to_index(hist[:,-k:],V)]; y=torch.multinomial(probs,1,generator=g).squeeze(1); hist=torch.cat([hist[:,1:],y[:,None]],1)
    states=[]; obs=[]; ys=[]; rem=n
    while rem>0:
        take=min(B,rem); h=hist[:take]; st=h[:,-k:].clone(); ob=h[:,-L:].clone(); probs=P[context_to_index(st,V)]
        y=torch.multinomial(probs,1,generator=g).squeeze(1); states.append(st.cpu()); obs.append(ob.cpu()); ys.append(y.cpu())
        probs_all=P[context_to_index(hist[:,-k:],V)]; ya=torch.multinomial(probs_all,1,generator=g).squeeze(1); hist=torch.cat([hist[:,1:],ya[:,None]],1); rem-=take
    return torch.cat(states).long(),torch.cat(obs).long(),torch.cat(ys).long()


In [ ]:
class TinyCausalTransformer(nn.Module):
    def __init__(self,V,max_context,d_model=256,n_layers=3,n_heads=8,dim_ff=512):
        super().__init__(); self.token_emb=nn.Embedding(V,d_model); self.pos_emb=nn.Embedding(max_context,d_model)
        layer=nn.TransformerEncoderLayer(d_model=d_model,nhead=n_heads,dim_feedforward=dim_ff,dropout=0.0,activation='gelu',batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=n_layers); self.norm=nn.LayerNorm(d_model); self.head=nn.Linear(d_model,V)
    def forward(self,x):
        B,L=x.shape; pos=torch.arange(L,device=x.device)[None,:]; h=self.token_emb(x)+self.pos_emb(pos)
        mask=torch.triu(torch.ones((L,L),dtype=torch.bool,device=x.device),diagonal=1); h=self.encoder(h,mask=mask); return self.head(self.norm(h[:,-1,:]))

EXPERIMENT_BY_K={2:'main_recovery',3:'main_recovery',4:'k4_refinement_12k'}
EXPECTED_STEPS={'main_recovery':3000,'k4_refinement_12k':12000}

def choose_checkpoint(rd):
    for n in ['checkpoint_best.pt','checkpoint_last.pt']:
        p=rd/n
        if p.exists(): return p
    raise FileNotFoundError(rd)

def discover_canonical_run(experiment,V,k,L,seed):
    root=PART1/'runs'/experiment; matches=[]
    for cp in root.glob('*/config.json'):
        try: cfg=json.loads(cp.read_text())
        except Exception: continue
        if int(cfg.get('V',-1))==V and int(cfg.get('k',-1))==k and int(cfg.get('L',-1))==L and int(cfg.get('seed',-1))==seed:
            matches.append((cp.parent,cfg))
    target=EXPECTED_STEPS[experiment]; exact=[x for x in matches if int(x[1].get('steps',-1))==target]
    if not exact: raise RuntimeError(f'No canonical run {experiment} k={k} L={L} seed={seed}; matches={len(matches)}')
    exact=sorted(exact,key=lambda x:str(x[0]))
    if len(matches)>1: print(f'[duplicate-safe] {len(matches)} identity matches -> {exact[0][0].name}')
    return exact[0]

def load_scratch(k,L,seed):
    exp=EXPERIMENT_BY_K[k]; rd,cfg=discover_canonical_run(exp,V,k,L,seed); P=trusted_torch_load(rd/'kernel.pt').float().to(DEVICE)
    ckpath=choose_checkpoint(rd); ck=trusted_torch_load(ckpath); state=ck['model'] if isinstance(ck,dict) and 'model' in ck else ck
    model=TinyCausalTransformer(V,int(state['pos_emb.weight'].shape[0]),int(cfg.get('d_model',256)),int(cfg.get('n_layers',3)),int(cfg.get('n_heads',8)),int(cfg.get('dim_ff',512))).to(DEVICE)
    model.load_state_dict(state,strict=True); model.eval(); return model,P,cfg,rd,ckpath


# A4 — Exact boundary and seed-stability audit

In [ ]:
a4_path=CSV_DIR/'A4_boundary_gap_seed_level.csv'
if a4_path.exists(): a4=pd.read_csv(a4_path)
else:
    rows=[]
    for k in KS:
        for seed in SCRATCH_SEEDS:
            _,P,cfg,rd,_=load_scratch(k,k,seed); info=exact_stationary_information(P,V,k)
            fm_path=rd/'final_metrics.json'; fm=json.loads(fm_path.read_text()) if fm_path.exists() else {}
            rows.append({'k':k,'seed':seed,'H_star_stationary':info['H_star'],'H_kminus1_stationary':info['H_km1'],'boundary_gap_stationary':info['gap'],'stationary_iters':info['stationary_iters'],'stationary_residual':info['stationary_residual'],'test_loss_Lk':fm.get('test_loss',np.nan),'model_gap_Lk':fm.get('test_loss',np.nan)-info['H_star'] if fm.get('test_loss') is not None else np.nan,'run_dir':str(rd)})
            del P; clean_gpu()
    a4=pd.DataFrame(rows); a4.to_csv(a4_path,index=False)
display(a4.sort_values(['k','seed']))
a4_summary=a4.groupby('k',as_index=False).agg(gap_min=('boundary_gap_stationary','min'),gap_mean=('boundary_gap_stationary','mean'),gap_median=('boundary_gap_stationary','median'),gap_max=('boundary_gap_stationary','max'),gap_sd=('boundary_gap_stationary','std'),model_gap_mean=('model_gap_Lk','mean'),model_gap_sd=('model_gap_Lk','std'),n=('seed','count'))
a4_summary.to_csv(CSV_DIR/'A4_boundary_gap_summary.csv',index=False); display(a4_summary)
assert (a4.boundary_gap_stationary>0).all(),'At least one sampled kernel has non-positive boundary gap.'
fig,ax=plt.subplots(figsize=(7.5,4.8))
for k in KS:
    d=a4[a4.k==k]; ax.scatter([k]*len(d),d.boundary_gap_stationary)
ax.set_xticks(KS); ax.set_xlabel('Markov order k'); ax.set_ylabel('H_{k-1} - H* (nats)'); ax.set_title('Information gap across source seeds'); fig.tight_layout(); fig.savefig(FIG_DIR/'A4_boundary_gap_by_seed.png',dpi=300); plt.show()


# A1 — Source-supported counterfactual intervention

In [ ]:
@torch.no_grad()
def model_probs(model,x): return torch.softmax(model(x).float(),dim=-1)
@torch.no_grad()
def oracle_probs(P,x,V,k): return P[context_to_index(x[:,-k:],V)].float()
@torch.no_grad()
def kl_pq(p,q):
    p=p.float().clamp_min(1e-30); q=q.float().clamp_min(1e-30); return (p*(torch.log(p)-torch.log(q))).sum(-1)
def uniform_alternative(cur,V,gen):
    r=torch.randint(0,V-1,cur.shape,generator=gen,device=cur.device); return r+(r>=cur).long()
@torch.no_grad()
def conditional_alternative(state,cond,V,gen):
    suffix=context_to_index(state[:,1:],V); probs=cond[suffix].clone(); probs.scatter_(1,state[:,:1],0.0); probs=probs/probs.sum(-1,keepdim=True).clamp_min(1e-30)
    alt=torch.multinomial(probs,1,generator=gen).squeeze(1); return alt,probs.gather(1,alt[:,None]).squeeze(1)
@torch.no_grad()
def run_a1_seed(model,P,k,seed):
    L=k+1; pi,_,_=stationary_distribution(P,V,k); cond=oldest_given_suffix(pi,V,k)
    sampler=PersistentMarkovSampler(P,V,k,A1_BATCH,L,8100000+10000*k+seed,CHAIN_WARMUP)
    gcnd=torch.Generator(device=DEVICE); gcnd.manual_seed(9200000+10000*k+seed)
    guni=torch.Generator(device=DEVICE); guni.manual_seed(9300000+10000*k+seed)
    rows=[]; seen=0
    while seen<A1_N_CONTEXTS:
        b=min(A1_BATCH,A1_N_CONTEXTS-seen)
        if b==A1_BATCH: x,_=sampler.sample(L)
        else: x,_=PersistentMarkovSampler(P,V,k,b,L,8100000+10000*k+seed+seen+17,CHAIN_WARMUP).sample(L)
        pf=oracle_probs(P,x,V,k); qf=model_probs(model,x); pos=L-k; state=x[:,-k:]
        ac,cp=conditional_alternative(state,cond,V,gcnd); xc=x.clone(); xc[:,pos]=ac; pc=oracle_probs(P,xc,V,k); qc=model_probs(model,xc)
        au=uniform_alternative(x[:,pos],V,guni); xu=x.clone(); xu[:,pos]=au; pu=oracle_probs(P,xu,V,k); qu=model_probs(model,xu)
        xi=x.clone(); xi[:,0]=uniform_alternative(x[:,0],V,guni); pi0=oracle_probs(P,xi,V,k)
        if (pi0-pf).abs().max().item()>1e-7: raise AssertionError('irrelevant intervention changed oracle')
        qi=model_probs(model,xi)
        ic=context_to_index(xc[:,-k:],V); iu=context_to_index(xu[:,-k:],V)
        rows.append(pd.DataFrame({'gain_supported':(kl_pq(pc,qf)-kl_pq(pc,qc)).cpu().numpy(),'gain_uniform':(kl_pq(pu,qf)-kl_pq(pu,qu)).cpu().numpy(),'gain_irrelevant':(kl_pq(pf,qf)-kl_pq(pf,qi)).cpu().numpy(),'cond_alt_prob':cp.cpu().numpy(),'stationary_prob_cf_supported':pi[ic].cpu().numpy(),'stationary_prob_cf_uniform':pi[iu].cpu().numpy(),'kl_before_supported':kl_pq(pc,qf).cpu().numpy(),'kl_after_supported':kl_pq(pc,qc).cpu().numpy()})); seen+=b
    d=pd.concat(rows,ignore_index=True); out={'k':k,'L':L,'seed':seed,'n_contexts':len(d)}
    for col in d.columns: out[col+'_mean']=float(d[col].mean()); out[col+'_median']=float(d[col].median())
    out['supported_vs_uniform_gain_diff']=out['gain_supported_mean']-out['gain_uniform_mean']; out['supported_kl_reduction_pct']=100*(out['kl_before_supported_mean']-out['kl_after_supported_mean'])/max(out['kl_before_supported_mean'],1e-12)
    return out


In [ ]:
a1_dir=OUT/'A1_checkpoints'; a1_dir.mkdir(exist_ok=True); rows=[]
for k in KS:
    for seed in SCRATCH_SEEDS:
        done=a1_dir/f'k{k}_seed{seed}.json'
        if done.exists(): rows.append(json.loads(done.read_text())); continue
        print('A1',k,seed); model,P,cfg,rd,ck=load_scratch(k,k+1,seed); r=run_a1_seed(model,P,k,seed); r.update({'run_dir':str(rd),'checkpoint':str(ck)}); save_json(done,r); rows.append(r); del model,P; clean_gpu()
a1=pd.DataFrame(rows); a1.to_csv(CSV_DIR/'A1_source_supported_counterfactual_seed_level.csv',index=False); display(a1)
a1_summary=a1.groupby('k',as_index=False).agg(supported_gain_mean=('gain_supported_mean','mean'),supported_gain_sd=('gain_supported_mean','std'),uniform_gain_mean=('gain_uniform_mean','mean'),irrelevant_gain_mean=('gain_irrelevant_mean','mean'),supported_reduction_pct_mean=('supported_kl_reduction_pct','mean'),cf_stationary_prob_supported=('stationary_prob_cf_supported_mean','mean'),cf_stationary_prob_uniform=('stationary_prob_cf_uniform_mean','mean'),n=('seed','count'))
a1_summary.to_csv(CSV_DIR/'A1_source_supported_counterfactual_summary.csv',index=False); display(a1_summary)


# Shared frozen-Qwen and probe utilities

In [ ]:
def choose_token_ids(model,V,seed):
    vs=model.config.vocab_size; special=set()
    for a in ['bos_token_id','eos_token_id','pad_token_id']:
        v=getattr(model.config,a,None)
        if v is not None: special.update(v if isinstance(v,list) else [int(v)])
    lo=min(1000,max(0,vs//20)); hi=max(lo+V+100,vs-1000); cand=np.array([i for i in range(lo,hi) if i not in special],dtype=np.int64)
    rng=np.random.default_rng(seed+91337); return torch.tensor(rng.choice(cand,V,replace=False),dtype=torch.long)
def map_symbols(x,mapping): return mapping[x]
def load_frozen(mid):
    print('Loading',mid); m=AutoModelForCausalLM.from_pretrained(mid,cache_dir=str(CACHE_DIR),torch_dtype=MODEL_DTYPE,low_cpu_mem_usage=True).to(DEVICE).eval()
    for p in m.parameters(): p.requires_grad_(False)
    return m
@torch.no_grad()
def extract_final(model,ids,batch):
    reps=[]
    for s in range(0,len(ids),batch):
        b=ids[s:s+batch].to(DEVICE); out=model(input_ids=b,attention_mask=torch.ones_like(b),output_hidden_states=True,use_cache=False,return_dict=True); reps.append(out.hidden_states[-1][:,-1,:].detach().cpu().to(torch.float16)); del out,b; clean_gpu()
    return torch.cat(reps)
class LinearState(nn.Module):
    def __init__(self,d,k,V): super().__init__(); self.k=k; self.V=V; self.h=nn.Linear(d,k*V)
    def forward(self,x): return self.h(x).view(-1,self.k,self.V)
class LinearNext(nn.Module):
    def __init__(self,d,V): super().__init__(); self.h=nn.Linear(d,V)
    def forward(self,x): return self.h(x)
class MLPNext(nn.Module):
    def __init__(self,d,V,width=None,depth=1):
        super().__init__(); h=width or min(1024,max(256,d)); layers=[nn.Linear(d,h),nn.GELU()]
        if depth>=2: layers += [nn.Linear(h,h),nn.GELU()]
        layers += [nn.Linear(h,V)]; self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)
def sloss(z,y): return F.cross_entropy(z.reshape(-1,z.shape[-1]),y.reshape(-1))
def train_probe(model,Xtr,ytr,Xva,yva,task,epochs,lr):
    model=model.to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=PROBE_WD); tr=DataLoader(TensorDataset(Xtr.float(),ytr),batch_size=PROBE_BATCH,shuffle=True); va=DataLoader(TensorDataset(Xva.float(),yva),batch_size=PROBE_BATCH); best=float('inf'); bs=None; bad=0
    for ep in range(epochs):
        model.train()
        for xb,yb in tr:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(set_to_none=True); z=model(xb); loss=sloss(z,yb) if task=='state' else F.cross_entropy(z,yb); loss.backward(); opt.step()
        model.eval(); tot=0
        with torch.no_grad():
            for xb,yb in va:
                xb,yb=xb.to(DEVICE),yb.to(DEVICE); z=model(xb); l=sloss(z,yb) if task=='state' else F.cross_entropy(z,yb); tot+=l.item()*len(xb)
        v=tot/len(Xva)
        if v<best-1e-5: best=v; bs={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=PATIENCE: break
    if bs: model.load_state_dict(bs)
    return model
@torch.no_grad()
def eval_state(model,X,y):
    dl=DataLoader(TensorDataset(X.float(),y),batch_size=PROBE_BATCH); exact=0; n=0; pos=None; ce=0
    for xb,yb in dl:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE); z=model(xb); p=z.argmax(-1); ce+=sloss(z,yb).item()*len(xb); c=(p==yb).sum(0).cpu(); pos=c if pos is None else pos+c; exact+=(p==yb).all(1).sum().item(); n+=len(xb)
    pa=(pos.float()/n).numpy(); return {'state_ce':ce/n,'state_exact_acc':exact/n,'state_mean_pos_acc':float(pa.mean()),'state_oldest_acc':float(pa[0])}
@torch.no_grad()
def eval_next(model,X,y):
    dl=DataLoader(TensorDataset(X.float(),y),batch_size=PROBE_BATCH); ce=0; cor=0; n=0
    for xb,yb in dl:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE); z=model(xb); ce+=F.cross_entropy(z,yb).item()*len(xb); cor+=(z.argmax(-1)==yb).sum().item(); n+=len(xb)
    return {'next_ce':ce/n,'next_acc':cor/n}
def onehot_state_features(s,V): return F.one_hot(s,num_classes=V).float().reshape(len(s),-1)


# A2 — Token-map robustness

In [ ]:
a2_dir=OUT/'A2_token_map_runs'; a2_dir.mkdir(exist_ok=True); a2_rows=[]
for mk,mid in MODELS.items():
    model=load_frozen(mid)
    for k in KS:
        P=generate_kernel(V,k,A2_SOURCE_SEED)
        for L in [max(1,k-1),k,k+1]:
            tr_s,tr_o,tr_y=generate_dataset(P,V,k,L,A2_N_TRAIN,A2_SOURCE_SEED+101); va_s,va_o,va_y=generate_dataset(P,V,k,L,A2_N_VAL,A2_SOURCE_SEED+202); te_s,te_o,te_y=generate_dataset(P,V,k,L,A2_N_TEST,A2_SOURCE_SEED+303)
            for ms in A2_MAP_SEEDS:
                done=a2_dir/f'{mk}_k{k}_L{L}_map{ms}.json'
                if done.exists(): a2_rows.append(json.loads(done.read_text())); continue
                print('A2',mk,k,L,ms); mapping=choose_token_ids(model,V,ms+int(hashlib.sha256(mk.encode()).hexdigest()[:6],16)); Xtr=extract_final(model,map_symbols(tr_o,mapping),EXTRACT_BATCH[mk]); Xva=extract_final(model,map_symbols(va_o,mapping),EXTRACT_BATCH[mk]); Xte=extract_final(model,map_symbols(te_o,mapping),EXTRACT_BATCH[mk]); d=Xtr.shape[1]
                sp=train_probe(LinearState(d,k,V),Xtr,tr_s,Xva,va_s,'state',LINEAR_EPOCHS,LINEAR_LR); met=eval_state(sp,Xte,te_s); res={'model_key':mk,'model_id':mid,'k':k,'L':L,'relative_context':L-k,'source_seed':A2_SOURCE_SEED,'map_seed':ms,'mapping':mapping.tolist(),**met}; save_json(done,res); a2_rows.append(res); del Xtr,Xva,Xte,sp; clean_gpu()
        del P
    del model; clean_gpu()
a2=pd.DataFrame(a2_rows); a2.to_csv(CSV_DIR/'A2_token_map_robustness_raw.csv',index=False)
a2_summary=a2.groupby(['model_key','k','L','relative_context'],as_index=False).agg(state_exact_mean=('state_exact_acc','mean'),state_exact_sd=('state_exact_acc','std'),state_exact_min=('state_exact_acc','min'),state_exact_max=('state_exact_acc','max'),oldest_mean=('state_oldest_acc','mean'),oldest_sd=('state_oldest_acc','std'),n_maps=('map_seed','nunique'))
a2_summary.to_csv(CSV_DIR/'A2_token_map_robustness_summary.csv',index=False); display(a2_summary)


# A3 — Probe capacity × training-set-size sweep

In [ ]:
A3_PROBES={'linear':lambda d,V:LinearNext(d,V),'mlp':lambda d,V:MLPNext(d,V,depth=1),'mlp_wide2':lambda d,V:MLPNext(d,V,width=min(2048,max(512,d*2)),depth=2)}
a3_dir=OUT/'A3_probe_sweep_runs'; a3_dir.mkdir(exist_ok=True); a3_rows=[]
for mk,mid in MODELS.items():
    model=load_frozen(mid)
    for k in A3_KS:
        L=k; P=generate_kernel(V,k,A3_SOURCE_SEED); tr_s,tr_o,tr_y=generate_dataset(P,V,k,L,A2_N_TRAIN,A3_SOURCE_SEED+101); va_s,va_o,va_y=generate_dataset(P,V,k,L,A2_N_VAL,A3_SOURCE_SEED+202); te_s,te_o,te_y=generate_dataset(P,V,k,L,A2_N_TEST,A3_SOURCE_SEED+303)
        mapping=choose_token_ids(model,V,A3_MAP_SEED+int(hashlib.sha256(mk.encode()).hexdigest()[:6],16)); Xtr=extract_final(model,map_symbols(tr_o,mapping),EXTRACT_BATCH[mk]); Xva=extract_final(model,map_symbols(va_o,mapping),EXTRACT_BATCH[mk]); Xte=extract_final(model,map_symbols(te_o,mapping),EXTRACT_BATCH[mk]); Str=onehot_state_features(tr_s,V); Sva=onehot_state_features(va_s,V); Ste=onehot_state_features(te_s,V)
        g=torch.Generator().manual_seed(77000+100*k); fullperm=torch.randperm(len(Xtr),generator=g)
        for frac in A3_FRACTIONS:
            n=max(128,int(len(Xtr)*frac)); perm=fullperm[:n]
            for pn,builder in A3_PROBES.items():
                for feat,TR,VA,TE in [('hidden',Xtr,Xva,Xte),('true_state',Str,Sva,Ste)]:
                    done=a3_dir/f'{mk}_k{k}_frac{frac}_{pn}_{feat}.json'
                    if done.exists(): a3_rows.append(json.loads(done.read_text())); continue
                    print('A3',mk,k,frac,pn,feat); epochs=LINEAR_EPOCHS if pn=='linear' else MLP_EPOCHS; lr=LINEAR_LR if pn=='linear' else MLP_LR; pb=train_probe(builder(TR.shape[1],V),TR[perm],tr_y[perm],VA,va_y,'next',epochs,lr); met=eval_next(pb,TE,te_y); res={'model_key':mk,'k':k,'L':L,'train_fraction':frac,'n_train':n,'probe':pn,'feature':feat,**met}; save_json(done,res); a3_rows.append(res); del pb; clean_gpu()
        del P,Xtr,Xva,Xte,Str,Sva,Ste; clean_gpu()
    del model; clean_gpu()
a3=pd.DataFrame(a3_rows); a3.to_csv(CSV_DIR/'A3_probe_capacity_data_sweep_raw.csv',index=False)
piv=a3.pivot_table(index=['model_key','k','train_fraction','probe'],columns='feature',values='next_ce').reset_index(); piv['hidden_minus_true_state_ce']=piv['hidden']-piv['true_state']; piv.to_csv(CSV_DIR/'A3_probe_capacity_data_sweep_contrasts.csv',index=False); display(piv.sort_values(['model_key','k','probe','train_fraction']))


# Final audit and export

In [ ]:
assert len(a1)==len(KS)*len(SCRATCH_SEEDS)
expected_a2=len(MODELS)*len(KS)*3*len(A2_MAP_SEEDS); assert len(a2)==expected_a2,(len(a2),expected_a2)
expected_a3=len(MODELS)*len(A3_KS)*len(A3_FRACTIONS)*len(A3_PROBES)*2; assert len(a3)==expected_a3,(len(a3),expected_a3)
assert (a4.boundary_gap_stationary>0).all()
for df in [a1,a2,a3,a4]:
    num=df.select_dtypes(include=[np.number]).to_numpy(float); assert np.isfinite(num).all()
audit={'status':'PASS','A1_rows':len(a1),'A2_rows':len(a2),'A3_rows':len(a3),'A4_rows':len(a4),'A4_all_boundary_gaps_positive':True,'A2_token_maps_per_configuration':len(A2_MAP_SEEDS),'A3_train_fractions':A3_FRACTIONS,'guardrails':['A1 supports robustness of native functional dependence under source-supported counterfactuals.','A2 concerns accessibility robustness only, not causal use.','A3 is a diagnostic of readout/data limitations.','A4 empirically verifies positive information gap for sampled kernels.']}
save_json(OUT/'reviewer_robustness_audit.json',audit); print(json.dumps(audit,indent=2))
(OUT/'README_outputs.md').write_text('# Reviewer Robustness Checks\nA1 source-supported counterfactuals\nA2 token-map robustness\nA3 probe sweep\nA4 boundary audit\n',encoding='utf-8')
zip_path=Path.cwd()/'Reviewer_Robustness_Checks_RESULTS.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file(): z.write(f,f.relative_to(OUT.parent))
print('RESULT ZIP:',zip_path)
